In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor


df = pd.read_csv('PHA_cleaned_data.csv')
df_cls = df.dropna(subset=['pha']).copy()

X = df_cls.drop(
    columns=[col for col in ['pha', 'diameter'] if col in df_cls.columns]
)
y_cls = df_cls['pha']

print(f'Classification Data Shape before preprocessing: {X.shape}')

# 1: ENCODING
cat_cols = X.select_dtypes(include=['object', 'category']).columns
if len(cat_cols) > 0:
  X[cat_cols] = OrdinalEncoder(
      handle_unknown='use_encoded_value', unknown_value=-1
  ).fit_transform(X[cat_cols])

# 2: HANDLING SKEWNESS
numerical_cols = X.select_dtypes(include=['number']).columns
skewness = X[numerical_cols].skew()
skewed_cols = skewness[abs(skewness) > 1.0].index

for col in skewed_cols:
  # Use log1p to safely handle zero or positive values
  X[col] = np.log1p(X[col])

# 3: SCALING
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 4: FEATURE IMPORTANCE & SELECTION
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=10)
X_cls_selected = pd.DataFrame(
    selector.fit_transform(X_scaled, y_cls),
    columns=X.columns[selector.get_support()]
)

print(f'Classification Data Shape after feature selection: {X_cls_selected.shape}')
print("Top 10 Selected Features:", list(X_cls_selected.columns))

Classification Data Shape before preprocessing: (938603, 34)
Classification Data Shape after feature selection: (938603, 10)
Top 10 Selected Features: ['neo', 'H', 'e', 'a', 'q', 'n', 'per', 'per_y', 'moid', 'moid_ld']


c:\Users\ponpriya\Desktop\Project_Third\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [4 5 6] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\ponpriya\Desktop\Project_Third\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [2]:
df = pd.read_csv('PHA_cleaned_data.csv')
df_reg = df.dropna(subset=['diameter']).copy()

X = df_reg.drop(
    columns=[col for col in ['pha', 'diameter'] if col in df_reg.columns]
)
y_reg = df_reg['diameter']

print(f'Regression Data Shape before preprocessing: {X.shape}')

# 1: ENCODING
cat_cols = X.select_dtypes(include=['object', 'category']).columns
if len(cat_cols) > 0:
  X[cat_cols] = OrdinalEncoder(
      handle_unknown='use_encoded_value', unknown_value=-1
  ).fit_transform(X[cat_cols])

# 2: HANDLING SKEWNESS
numerical_cols = X.select_dtypes(include=['number']).columns
skewness = X[numerical_cols].skew()
skewed_cols = skewness[abs(skewness) > 1.0].index

for col in skewed_cols:
  X[col] = np.log1p(X[col])

# 3: SCALING
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 4: FEATURE IMPORTANCE & SELECTION
import numpy as np
import joblib

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_scaled, y_reg)

importances = rf_reg.feature_importances_
top_indices = np.argsort(importances)[::-1][:12]

X_reg_selected = X_scaled.iloc[:, top_indices]

print(f'Regression Data Shape after feature selection: {X_reg_selected.shape}')
print("Top 10 Selected Regression Features:", list(X_reg_selected.columns))

joblib.dump(X_reg_selected, 'X_reg_selected.pkl')
print("[INFO] X_reg_selected saved successfully as 'X_reg_selected.pkl'!")

Regression Data Shape before preprocessing: (136209, 34)
Regression Data Shape after feature selection: (136209, 12)
Top 10 Selected Regression Features: ['H', 'albedo', 'rms', 'q', 'moid', 'moid_ld', 'sigma_n', 'diameter_sigma', 'per', 'n', 'ma', 'a']
[INFO] X_reg_selected saved successfully as 'X_reg_selected.pkl'!


In [3]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import pandas as pd
import numpy as np

df = pd.read_csv('PHA_cleaned_data.csv')

# 3 FEATURE SELECTION
orbital_features = ['a', 'e', 'i', 'per', 'moid_ld']

selected_cols = [col for col in orbital_features if col in df.columns]
X = df[selected_cols].copy()

print(f'Clustering Data Shape (Orbital Elements only): {X.shape}')

# 1: ENCODING
cat_cols = X.select_dtypes(include=['object', 'category']).columns
if len(cat_cols) > 0:
    X[cat_cols] = OrdinalEncoder(
        handle_unknown='use_encoded_value', unknown_value=-1
    ).fit_transform(X[cat_cols])

# 2: HANDLING SKEWNESS
numerical_cols = X.select_dtypes(include=['number']).columns
skewness = X[numerical_cols].skew()
skewed_cols = skewness[abs(skewness) > 1.0].index

for col in skewed_cols:
    X[col] = np.log1p(X[col])

# 3: SCALING
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 4: FEATURE SELECTION
selector = VarianceThreshold(threshold=0.01)
selector.fit(X_scaled)
X_cluster_selected = X_scaled.loc[:, selector.get_support()]

print(f'Final Clustering Data Shape: {X_cluster_selected.shape}')

Clustering Data Shape (Orbital Elements only): (958524, 5)
Final Clustering Data Shape: (958524, 5)


In [4]:
import joblib

# 1. Classification Data
joblib.dump(X_cls_selected, 'X_cls_selected.pkl')
joblib.dump(y_cls, 'y_cls.pkl')

# 2. Regression Data (y is already right there in your environment!)
joblib.dump(X_reg_selected, 'X_reg_selected.pkl')
joblib.dump(y_reg, 'y_reg.pkl')

# 3. Clustering Data
joblib.dump(X_cluster_selected, 'X_cluster_selected.pkl')

print("Datasets saved successfully!")

Datasets saved successfully!
